# AI vs Human Text Classification & Detection using Sentence Embeddings: Model Benchmark

Dataset : https://www.kaggle.com/datasets/hasanyiitakbulut/ai-and-human-text-dataset

#  Step 0 - import library 

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



# Step 1 - Load dataset

In [ ]:
dataset =  pd.read_csv("/kaggle/input/datasets/hasanyiitakbulut/ai-and-human-text-dataset/data_for_preprocessing.csv")
dataset.head(10) # pring 10 samples

In [ ]:
dataset.info()
dataset.describe()

# Step 2 - Data Preprocessing

In [ ]:
print(f"duplicate count: {dataset.duplicated().sum()}") # check for duplicate 
dataset.isnull().sum() # check for miision values

In [ ]:
# to see how the data is balance
sns.countplot(data=dataset, x="Author",stat="percent")
plt.show()

In [ ]:
# Count top 10 words - to see if leakage e.g "generated", "AI" text is being mensions in the data
word_freq = Counter(" ".join(dataset["Text"]).split()).most_common(10)

# Unzip into words and counts
words, counts = zip(*word_freq)

# Plot
sns.barplot(x=counts, y=words)
plt.title("Top 20 Most Frequent Words")
plt.show()

In [ ]:
print(f"Largest text in 'Text column' has : {dataset['Text'].astype(str).map(len).max()} count") # largest sentence length in 'text' column
dataset["text_length"] = dataset["Text"].str.split().str.len()
sns.histplot(dataset["text_length"], bins=50)
plt.title("Text Length Distribution")
plt.xlabel("Number of Words")
plt.show()

In [ ]:
# encode the terget lebel
le = LabelEncoder()
dataset["Author"] = le.fit_transform(dataset["Author"])
dataset.head()

In [ ]:
X = dataset['Text']
y = dataset['Author']

In [ ]:
# spliting the dataset 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Step 3  - Text Embeddings 

In [ ]:
# Create meaningful numerical embeddings of the texts.

# Check if GPU is available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# 3. Convert text to embeddings
X_train_embeddings = model.encode(X_train.tolist(), show_progress_bar=True, batch_size=128)
X_test_embeddings = model.encode(X_test.tolist(), show_progress_bar=True, batch_size=128)


In [ ]:
X_train_embeddings.shape, X_test_embeddings.shape


In [ ]:
# 1. Reduce dimensions to 2D
tsne = TSNE(n_components=2, random_state=42)
X_2d = tsne.fit_transform(X_train_embeddings)

tsne_comment = """
TSNE Graph Interpretation Notes

1 - Overlapping dots → the data points are very similar (model can’t clearly distinguish them)

2 - Tight clusters → points in the same group share strong similarity (same class or pattern)

3 - Dots far away from their cluster → likely outliers or unusual samples

4 - Well-separated clusters → classes are easily distinguishable (good for classification)

5 - Mixed clusters (colors blended) → features are not strong enough to separate classes
"""

print(tsne_comment)
# 2. Plot it
plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_2d[:,0], y=X_2d[:,1], hue=y_train, palette='viridis', alpha=0.6)
plt.title("Visualizing Sentence Clusters with t-SNE")
plt.show()


In [ ]:
iso = IsolationForest(contamination=0.05)

# Train on training data
iso.fit(X_train_embeddings)

# Predict on test data
labels = iso.predict(X_test_embeddings)

# Get outliers from test set
outliers = X_test[labels == -1]

# Get correct indices relative to test set
outlier_idx = np.where(labels == -1)[0]

# Print actual text from TEST set (outliers)
#print(X_test.iloc[outlier_idx])
print(f"found :{len(X_test.iloc[outlier_idx])} outliers")

In [ ]:
#manual visulaization of outliers
print(X_test[2797])
print(X_test[5946])
print(X_test[2989])

# remove outliers
#clean_X = X_test[labels != -1]

# Step 4 - Modeling, Evaluation & Benchmarking

In [ ]:
# models init
models = {
    "Logistic Regression": LogisticRegression(max_iter=100),
    "SVM": LinearSVC(dual='auto'),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(n_estimators=200, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=200, verbose=-1)
}


results = []

for name, model in models.items():
    #Cross-validation on training set (F1 score)
    cv_scores = cross_val_score(model, X_train_embeddings, y_train, cv=5, scoring="f1")
    mean_cv = cv_scores.mean()
    
    #training
    model.fit(X_train_embeddings, y_train)
    
    #Predict on test
    preds = model.predict(X_test_embeddings)
    
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, preds), 3),
        "CV F1": round(mean_cv, 3),  # mean CV score
        "Precision": round(precision_score(y_test, preds), 3),
        "Recall": round(recall_score(y_test, preds), 3),
        "F1 Score": round(f1_score(y_test, preds), 3)
    })



In [ ]:
# Make DataFrame , sort by F1 Score
results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)

# Apply style: blue-white gradient
styled_df = (results_df.style.background_gradient(cmap="Blues"))

styled_df